In [2]:
import argparse
from itertools import product

import numpy as np

import sys
sys.path.append("../")
from cc2cc.utils import gen_mole
from cc2cc.utils.env_var import DATA_PATH
from cc2cc.utils.parser import gen_name_args

origin_mol_str_list = [
    "molecule0-W4_11",
    # "molecule1-W4_11",
    # "molecule2-W4_11",
    # "molecule3-W4_11",
    # "molecule4-W4_11",
    # "molecule5-W4_11",
]
name_mol_str_list = [
    "molecule0",
    # "molecule1",
    # "molecule2",
    # "molecule3",
    # "molecule4",
    # "molecule5",
    # "molecule6",
]
name_mol_str_exclude_list = [
]

mol_elements_dict = {}
len_elements_dict = {}

if __name__ == "__main__":
    error_molecule = []

    name_mol_list = gen_name_args(name_mol_str_list, "gmtkn-def2")
    name_mol_exclude_list = gen_name_args(
        name_mol_str_exclude_list, "gmtkn-def2", if_exclude=True
    )
    name_mol_list = [mol for mol in name_mol_list if mol not in name_mol_exclude_list]

    origin_mol_list = gen_name_args(origin_mol_str_list, "gmtkn-def2")
    name_mol_list = origin_mol_list + name_mol_list

    error_molecule = []
    print(f"Name Molecule List: {name_mol_list}")

    for name_mol in name_mol_list:
        try:
            mol = gen_mole(
                name_mol,
                0,
                1,
                0,
                "def2-TZVPD",
                "gmtkn-def2",
                if_rotate=True,
                if_rotate_random=False,
                solve_symmetry=True,
                verbose=1,
            )

            mol_elements = list(np.array(mol.elements))
            mol_atom_coords = list(mol.atom_coords())
            mol_atom_coords.append(name_mol)
            # print(mol_atom_coords)
            mol_elements.extend([mol.charge, mol.spin])
            mol_elements_str = "-".join(map(str, mol_elements))
            if mol_elements_str not in mol_elements_dict:
                mol_elements_dict[mol_elements_str] = [mol_atom_coords]
            else:
                mol_elements_dict[mol_elements_str].append(mol_atom_coords)
            len_elements_dict[mol_elements_str] = len(list(np.array(mol.elements)))

        except (ValueError, RuntimeError) as e:
            print(f"ERROR: {name_mol}")
            print(e)
            error_molecule.append(name_mol)
            print(f"Error molecule: {error_molecule}")
        finally:
            print(f"Processed: {name_mol}")
        print()

    print(f"Error molecule: {error_molecule}")

Name Molecule List: ['W4_11-al', 'W4_11-b', 'W4_11-be', 'W4_11-c', 'W4_11-cl', 'W4_11-f', 'W4_11-h', 'W4_11-n', 'W4_11-o', 'W4_11-p', 'W4_11-s', 'W4_11-si', 'ADDON_As', 'ADDON_Bi', 'ADDON_Ge', 'ADDON_I', 'ADDON_Pb', 'ADDON_Sb', 'ADDON_Se', 'ADDON_Te', 'AHB21-1A', 'AHB21-2A', 'AHB21-3A', 'AHB21-4A', 'AHB21-5A', 'AHB21-6A', 'AHB21-7A', 'AHB21-8A', 'ALK8-li+', 'ALK8-na+', 'ALKBDE10-be', 'ALKBDE10-ca', 'ALKBDE10-f', 'ALKBDE10-h', 'ALKBDE10-k', 'ALKBDE10-li', 'ALKBDE10-mg', 'ALKBDE10-na', 'ALKBDE10-o', 'ALKBDE10-s', 'BH76-O', 'BH76-cl', 'BH76-cl-', 'BH76-f', 'BH76-f-', 'BH76-h', 'CHB6-22A', 'CHB6-23A', 'CHB6-24A', 'CHB6-25A', 'CHB6-26A', 'CHB6-27A', 'DC13-be', 'DIPCS10-be', 'DIPCS10-be_2+', 'DIPCS10-mg', 'DIPCS10-mg_2+', 'G21EA-EA_c', 'G21EA-EA_c-', 'G21EA-EA_cl', 'G21EA-EA_cl-', 'G21EA-EA_f', 'G21EA-EA_f-', 'G21EA-EA_o', 'G21EA-EA_o-', 'G21EA-EA_p', 'G21EA-EA_p-', 'G21EA-EA_s', 'G21EA-EA_s-', 'G21EA-EA_si', 'G21EA-EA_si-', 'G21IP-al', 'G21IP-al+', 'G21IP-b', 'G21IP-b+', 'G21IP-be', 'G21IP-

In [5]:
for mol_elements_name, mol_elements in mol_elements_dict.items():
    # print(f"Processing {len_elements_dict[mol_elements_name]}")
    # print(f"{mol_elements}")

    identifiables = [0]
    for i_elements in range(1, len(mol_elements)):
        distance_list = np.zeros(len(identifiables))
        for iter, identifiable in enumerate(identifiables):
            for i_element in range(len(mol_elements[i_elements]) - 1):
                distance_list[iter] = max(
                    np.linalg.norm(
                        mol_elements[i_elements][i_element]
                        - mol_elements[identifiable][i_element]
                    ),
                    distance_list[iter],
                )
        if np.all(distance_list > 0.2):
            identifiables.append(i_elements)
        else:
            print(f"Merging {mol_elements[i_elements][-1]} to {mol_elements[identifiables[np.argmin(distance_list)]][-1]}")
        # else:
        #     print(f"Skipping {mol_elements[i_elements][-1]}")

    # print("===identifiables===")
    for identifiable in identifiables:
        # if mol_elements[identifiable][-1].startswith("W4_11"):
        #     continue
        print(f'"{mol_elements[identifiable][-1]}",')
        # print(
        #     f"{np.array2string(np.array(mol_elements[identifiable][:-1]), formatter={'float': '{: .2f}'.format})}"
        # )
    print()

Merging G21IP-al to W4_11-al
Merging W4_11-al to W4_11-al
"W4_11-al",

Merging G21IP-b to W4_11-b
Merging W4_11-b to W4_11-b
"W4_11-b",

Merging ALKBDE10-be to W4_11-be
Merging DC13-be to W4_11-be
Merging DIPCS10-be to W4_11-be
Merging G21IP-be to W4_11-be
Merging W4_11-be to W4_11-be
"W4_11-be",

Merging G21EA-EA_c to W4_11-c
Merging G21IP-c to W4_11-c
Merging W4_11-c to W4_11-c
"W4_11-c",

Merging BH76-cl to W4_11-cl
Merging G21EA-EA_cl to W4_11-cl
Merging G21IP-cl to W4_11-cl
Merging HEAVYSB11-cl to W4_11-cl
Merging W4_11-cl to W4_11-cl
"W4_11-cl",

Merging ALKBDE10-f to W4_11-f
Merging BH76-f to W4_11-f
Merging G21EA-EA_f to W4_11-f
Merging G21IP-f to W4_11-f
Merging W4_11-f to W4_11-f
"W4_11-f",

Merging ALKBDE10-h to W4_11-h
Merging BH76-h to W4_11-h
Merging G21IP-h to W4_11-h
Merging RC21-3p2 to W4_11-h
Merging SIE4x4-h to W4_11-h
Merging W4_11-h to W4_11-h
"W4_11-h",

Merging G21IP-n to W4_11-n
Merging W4_11-n to W4_11-n
"W4_11-n",

Merging ALKBDE10-o to W4_11-o
Merging BH76-O 

In [8]:
mol_elements_dict

{'Be-Cl-Cl-0-0': [[array([0., 0., 0.]),
   array([-3.40120467,  0.        ,  0.        ]),
   array([3.40120467, 0.        , 0.        ]),
   'W4_11-becl2']],
 'Be-F-F-0-0': [[array([0., 0., 0.]),
   array([-2.60704726,  0.        ,  0.        ]),
   array([2.60704726, 0.        , 0.        ]),
   'W4_11-bef2']],
 'C-Cl-Cl-0-0': [[array([0.       , 1.6089375, 0.       ]),
   array([-2.64638569, -0.27252296,  0.        ]),
   array([ 2.64638569, -0.27252296,  0.        ]),
   'W4_11-ccl2']],
 'C-F-F-0-0': [[array([ 0.        , -1.13580973,  0.        ]),
   array([-1.94261956,  0.35899387,  0.        ]),
   array([1.94261956, 0.35899387, 0.        ]),
   'W4_11-cf2']],
 'O-Cl-Cl-0-0': [[array([ 0.        , -1.48572494,  0.        ]),
   array([-2.64783133,  0.33525673,  0.        ]),
   array([2.64783133, 0.33525673, 0.        ]),
   'W4_11-cl2o']],
 'C-N-Cl-0-0': [[array([1.28237933, 0.        , 0.        ]),
   array([3.47820707, 0.        , 0.        ]),
   array([-1.80865703,  0.   